In [ ]:
# Import libraries and set configuration parameters
import os
import gc
import time
import sys
import numpy as np
import pandas as pd
import json
import datetime
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Set optimal parameters for i7 processor with 64GB RAM
CHUNK_SIZE = 75000    # Larger chunks to utilize more RAM
OVERLAP = 1000        # Consistent overlap for context preservation
MAX_WORKERS = 6       # Balance between parallelism and system resources


In [ ]:
def clear_memory():
    """Force a complete garbage collection to free memory"""
    import gc
    print("Clearing memory...")
    gc.collect(generation=2)  # Full collection
    if hasattr(gc, 'unfreeze'):  # PyPy specific
        gc.unfreeze()
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("Cleared CUDA cache")
    except (ImportError, AttributeError):
        pass


In [ ]:
def init_worker():
    """Initialize each worker process with required NLP models"""
    print(f"Initializing worker process {os.getpid()}")
    
    # Load models in each worker process to avoid sharing issues
    global nlp_sm, nlp_lg
    try:
        import spacy
        nlp_sm = spacy.load("en_core_web_sm")
        print(f"Worker {os.getpid()} loaded spaCy small model")
        
        nlp_lg = spacy.load("en_core_web_lg")
        if 'spacytextblob' not in nlp_lg.pipe_names:
            from spacytextblob.spacytextblob import SpacyTextBlob
            nlp_lg.add_pipe("spacytextblob")
        print(f"Worker {os.getpid()} loaded spaCy large model with SpacyTextBlob")
    except Exception as e:
        print(f"Worker {os.getpid()} error loading spaCy models: {e}")


In [ ]:
def ner_spacy(text):
    """Extract named entities using spaCy"""
    start_time = time.time()
    doc = nlp_lg(text)
    
    # Extract PERSON entities as potential characters
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ == "PERSON"]
    
    # Count unique character names
    unique_characters = set(entity[0] for entity in entities)
    
    end_time = time.time()
    return {
        "entities": entities,
        "unique_characters": list(unique_characters),
        "time_taken": end_time - start_time,
        "count": len(unique_characters)
    }

def ner_nltk(text):
    """Extract named entities using NLTK"""
    start_time = time.time()
    
    entities = []
    sentences = nltk.sent_tokenize(text)
    
    for sent in sentences:
        tokens = nltk.word_tokenize(sent)
        tagged = nltk.pos_tag(tokens)
        chunks = nltk.ne_chunk(tagged)
        
        # Extract person entities
        for chunk in chunks:
            if hasattr(chunk, 'label') and chunk.label() == 'PERSON':
                name = ' '.join(c[0] for c in chunk)
                entities.append((name, 'PERSON'))
    
    # Count unique character names
    unique_characters = set(entity[0] for entity in entities)
    
    end_time = time.time()
    return {
        "entities": entities,
        "unique_characters": list(unique_characters),
        "time_taken": end_time - start_time,
        "count": len(unique_characters)
    }


In [ ]:
def sentiment_spacy(text):
    """Analyze sentiment using spaCy with SpacyTextBlob"""
    start_time = time.time()
    
    # Using SpacyTextBlob for sentiment analysis
    doc = nlp_lg(text)
    
    # Get sentence-level sentiment
    sentences = [sent.text for sent in doc.sents]
    emotional_sentences = []
    
    for sent in sentences:
        sent_doc = nlp_lg(sent)
        # SpacyTextBlob provides polarity and subjectivity
        polarity = sent_doc._.blob.polarity
        subjectivity = sent_doc._.blob.subjectivity
        
        # Consider sentences with non-zero polarity and high subjectivity
        if abs(polarity) > 0.1 and subjectivity > 0.3:
            emotional_sentences.append((sent, polarity))
    
    # Sort by emotional intensity (absolute value of polarity)
    emotional_sentences.sort(key=lambda x: abs(x[1]), reverse=True)
    
    end_time = time.time()
    return {
        "emotional_sentences": emotional_sentences[:5],  # Top 5 most emotional
        "time_taken": end_time - start_time,
        "count": len(emotional_sentences)
    }

def sentiment_nltk(text):
    """Analyze sentiment using NLTK's VADER"""
    start_time = time.time()
    sid = SentimentIntensityAnalyzer()
    
    # Split text into paragraphs or sentences
    sentences = nltk.sent_tokenize(text)
    sentiment_scores = [sid.polarity_scores(sentence) for sentence in sentences]
    
    # Find most emotional sentences (highest absolute compound score)
    sentence_sentiments = [(sentences[i], scores["compound"]) for i, scores in enumerate(sentiment_scores)]
    emotional_sentences = sorted(sentence_sentiments, key=lambda x: abs(x[1]), reverse=True)[:5]
    
    end_time = time.time()
    return {
        "emotional_sentences": emotional_sentences,
        "time_taken": end_time - start_time,
        "count": len(sentence_sentiments)
    }


In [ ]:
def speed_spacy(text):
    """Measure processing speed using spaCy"""
    start_time = time.time()
    doc = nlp_lg(text)
    
    # Perform some standard NLP operations
    _ = [token.text for token in doc]  # Tokenization
    _ = [token.pos_ for token in doc]  # POS tagging
    _ = [token.dep_ for token in doc]  # Dependency parsing
    _ = [ent for ent in doc.ents]      # NER
    
    end_time = time.time()
    processing_time = end_time - start_time
    chars_per_second = len(text) / processing_time if processing_time > 0 else 0
    
    return {
        "time_taken": processing_time,
        "chars_per_second": chars_per_second,
        "text_length": len(text)
    }

def speed_nltk(text):
    """Measure processing speed using NLTK"""
    start_time = time.time()
    
    # Perform standard NLP operations
    sentences = nltk.sent_tokenize(text)
    all_tokens = []
    for sent in sentences:
        tokens = nltk.word_tokenize(sent)
        all_tokens.extend(tokens)
    
    # Limit to first 10,000 tokens for POS tagging to avoid excessive time
    tagged = nltk.pos_tag(all_tokens[:10000])
    
    end_time = time.time()
    processing_time = end_time - start_time
    chars_per_second = len(text) / processing_time if processing_time > 0 else 0
    
    return {
        "time_taken": processing_time,
        "chars_per_second": chars_per_second,
        "text_length": len(text)
    }


In [ ]:
def benchmark_performance(text, task="all", libraries=None):
    """
    Benchmark NLP tasks performance on the given text
    
    Args:
        text (str): Text to process
        task (str): Task type or "all" for all tasks
        libraries (list): List of libraries to benchmark
        
    Returns:
        dict: Results organized by task and library
    """
    if libraries is None:
        libraries = ["spacy", "nltk"]
    
    results = {}
    
    # Determine which tasks to run
    tasks = []
    if task == "all":
        tasks = ["ner", "sentiment", "dialogue_detection", "processing_speed"]
    elif task == "minimal":
        tasks = ["processing_speed"]  # Fallback option for memory issues
    else:
        tasks = [task]
    
    # Run each selected task
    for t in tasks:
        results[t] = {}
        
        if t == "ner":
            # Named Entity Recognition
            if "spacy" in libraries:
                try:
                    results[t]["spacy"] = ner_spacy(text)
                except Exception as e:
                    results[t]["spacy"] = {"time_taken": 0, "error": str(e)}
            
            if "nltk" in libraries:
                try:
                    results[t]["nltk"] = ner_nltk(text)
                except Exception as e:
                    results[t]["nltk"] = {"time_taken": 0, "error": str(e)}
                    
        elif t == "sentiment":
            # Sentiment Analysis
            if "spacy" in libraries:
                try:
                    results[t]["spacy"] = sentiment_spacy(text)
                except Exception as e:
                    results[t]["spacy"] = {"time_taken": 0, "error": str(e)}
            
            if "nltk" in libraries:
                try:
                    results[t]["nltk"] = sentiment_nltk(text)
                except Exception as e:
                    results[t]["nltk"] = {"time_taken": 0, "error": str(e)}
                    
        elif t == "dialogue_detection":
            # This would be your dialogue detection functions
            # For now, using placeholder dummy data
            if "spacy" in libraries:
                results[t]["spacy"] = {
                    "time_taken": 0.1, 
                    "count": 5,
                    "detected_dialogue": ["Hello there.", "How are you?", "I'm fine."]
                }
            
            if "nltk" in libraries:
                results[t]["nltk"] = {
                    "time_taken": 0.15, 
                    "count": 4,
                    "detected_dialogue": ["Hello there.", "How are you?"]
                }
                
        elif t == "processing_speed":
            # Processing Speed
            if "spacy" in libraries:
                try:
                    results[t]["spacy"] = speed_spacy(text)
                except Exception as e:
                    results[t]["spacy"] = {"time_taken": 0, "error": str(e)}
            
            if "nltk" in libraries:
                try:
                    results[t]["nltk"] = speed_nltk(text)
                except Exception as e:
                    results[t]["nltk"] = {"time_taken": 0, "error": str(e)}
    
    return results


In [ ]:
def process_sample_in_chunks(sample_id, sample_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """
    Process a sample text in smaller, overlapping chunks to improve performance
    
    Args:
        sample_id (str): Identifier for the sample
        sample_text (str): The full text to process
        chunk_size (int): Size of each chunk in characters
        overlap (int): Number of overlapping characters between chunks
        
    Returns:
        list: Processed results from all chunks with metadata
    """
    print(f"Processing {sample_id} (length: {len(sample_text)} chars, PID: {os.getpid()})")
    
    # For very small samples, just process the entire text
    if len(sample_text) <= chunk_size:
        try:
            return process_sample(sample_id, sample_text)
        except Exception as e:
            print(f"Error processing {sample_id}: {str(e)}")
            return [{
                "sample_id": sample_id,
                "error": str(e),
                "sample_length": len(sample_text)
            }]
    
    # For larger texts, split into chunks with some overlap
    chunks = []
    chunk_ids = []
    
    # Split the text into overlapping chunks
    for i in range(0, len(sample_text), chunk_size - overlap):
        chunk_end = min(i + chunk_size, len(sample_text))
        chunks.append(sample_text[i:chunk_end])
        chunk_ids.append(f"{sample_id}#chunk{len(chunks)}")
    
    print(f"Split into {len(chunks)} chunks")
    
    # Process each chunk and collect results
    sample_results = []
    
    for i, (chunk, chunk_id) in enumerate(tqdm(zip(chunks, chunk_ids), 
                                              total=len(chunks), 
                                              desc=f"Processing {sample_id}")):
        try:
            chunk_results = process_single_chunk(sample_id, chunk_id, chunk, i+1, len(chunks), len(sample_text))
            sample_results.extend(chunk_results)
            
            # Force garbage collection after each chunk
            gc.collect()
        except Exception as e:
            print(f"Error processing chunk {chunk_id}: {str(e)}")
            sample_results.append({
                "sample_id": sample_id,
                "chunk_id": chunk_id,
                "chunk_num": i+1,
                "chunk_size": len(chunk),
                "sample_length": len(sample_text),
                "error": str(e)
            })
    
    # Deduplicate and merge results
    return merge_chunk_results(sample_results)


In [ ]:
def process_single_chunk(sample_id, chunk_id, chunk, chunk_num, total_chunks, sample_length):
    """
    Process a single chunk and return the results
    
    Args:
        sample_id (str): Identifier for the sample
        chunk_id (str): Identifier for this chunk
        chunk (str): Text content of the chunk
        chunk_num (int): Number of this chunk in sequence
        total_chunks (int): Total number of chunks for this sample
        sample_length (int): Length of the entire sample text
        
    Returns:
        list: Results for this chunk with metadata
    """
    print(f"  - Processing chunk {chunk_num}/{total_chunks} ({len(chunk)} chars)")
    
    # For each chunk, run NLP tasks
    try:
        results = benchmark_performance(
            text=chunk,
            task="all",
            libraries=["spacy", "nltk"]
        )
    except MemoryError:
        gc.collect()  # Try to free memory
        # Retry with reduced operations if possible
        print(f"  ! Memory error in chunk {chunk_num}, retrying with minimal processing")
        results = benchmark_performance(
            text=chunk,
            task="minimal",  # Lighter processing mode
            libraries=["nltk"]  # Use just one lighter library
        )
    except Exception as e:
        print(f"  ! Error in chunk {chunk_num}: {str(e)}")
        return [{
            "sample_id": sample_id,
            "chunk_id": chunk_id,
            "chunk_num": chunk_num,
            "chunk_size": len(chunk),
            "sample_length": sample_length,
            "error": str(e)
        }]
    
    # Collect results
    chunk_results = []
    for task, lib_results in results.items():
        for lib, res in lib_results.items():
            result_row = {
                "sample_id": sample_id,
                "chunk_id": chunk_id,
                "chunk_num": chunk_num,
                "chunk_size": len(chunk),
                "sample_length": sample_length,
                "task": task,
                "library": lib,
                "time_taken": res["time_taken"],
                "chunk_position": "middle"
            }
            
            # Mark first and last chunks (useful for merging results)
            if chunk_num == 1:
                result_row["chunk_position"] = "first"
            elif chunk_num == total_chunks:
                result_row["chunk_position"] = "last"
            
            # Add task-specific metrics
            if task == "dialogue_detection" and "count" in res:
                result_row["count"] = res["count"]
                result_row["sample_dialogues"] = res["detected_dialogue"][:3] if res["detected_dialogue"] else []
            elif task == "ner" and "count" in res:
                result_row["unique_count"] = res["count"]
                result_row["characters"] = res["unique_characters"][:10] if res["unique_characters"] else []
            elif task == "sentiment" and "count" in res:
                result_row["emotional_count"] = res["count"]
                result_row["emotional_sentences"] = res["emotional_sentences"][:3] if res["emotional_sentences"] else []
            elif task == "processing_speed" and "chars_per_second" in res:
                result_row["chars_per_second"] = res["chars_per_second"]
            
            chunk_results.append(result_row)
    
    return chunk_results

def process_sample(sample_id, sample_text):
    """
    Process a small sample without chunking
    
    Args:
        sample_id (str): Identifier for the sample
        sample_text (str): The text to process
        
    Returns:
        list: Results with metadata
    """
    try:
        results = benchmark_performance(
            text=sample_text,
            task="all",
            libraries=["spacy", "nltk"]
        )
        
        # Format results
        sample_results = []
        for task, lib_results in results.items():
            for lib, res in lib_results.items():
                result_row = {
                    "sample_id": sample_id,
                    "sample_length": len(sample_text),
                    "task": task,
                    "library": lib,
                    "time_taken": res["time_taken"]
                }
                
                # Add task-specific metrics
                if task == "dialogue_detection" and "count" in res:
                    result_row["count"] = res["count"]
                    result_row["sample_dialogues"] = res["detected_dialogue"][:3] if res["detected_dialogue"] else []
                elif task == "ner" and "count" in res:
                    result_row["unique_count"] = res["count"]
                    result_row["characters"] = res["unique_characters"][:10] if res["unique_characters"] else []
                elif task == "sentiment" and "count" in res:
                    result_row["emotional_count"] = res["count"]
                    result_row["emotional_sentences"] = res["emotional_sentences"][:3] if res["emotional_sentences"] else []
                elif task == "processing_speed" and "chars_per_second" in res:
                    result_row["chars_per_second"] = res["chars_per_second"]
                
                sample_results.append(result_row)
        
        return sample_results
    
    except Exception as e:
        print(f"Error processing sample {sample_id}: {str(e)}")
        return [{
            "sample_id": sample_id,
            "error": str(e),
            "sample_length": len(sample_text)
        }]


In [ ]:
def merge_chunk_results(results):
    """
    Merge and deduplicate results from overlapping chunks
    
    Args:
        results (list): List of result dictionaries from chunks
        
    Returns:
        list: Merged results with duplicates removed
    """
    if not results:
        return []
    
    # Group results by task and library
    grouped = defaultdict(list)
    
    for res in results:
        if "error" in res:
            # Keep error results separate
            continue
            
        key = (res["sample_id"], res["task"], res["library"])
        grouped[key].append(res)
    
    merged_results = []
    
    # Handle errors separately
    error_results = [r for r in results if "error" in r]
    merged_results.extend(error_results)
    
    # Process each task+library group
    for key, group_results in grouped.items():
        sample_id, task, library = key
        
        # Sort by chunk number
        group_results.sort(key=lambda x: x["chunk_num"])
        
        # Create a base record
        base_record = {
            "sample_id": sample_id,
            "task": task,
            "library": library,
            "sample_length": group_results[0]["sample_length"],
            "chunks_processed": len(group_results),
            "merged": True
        }
        
        # Merge time metrics
        total_time = sum(r["time_taken"] for r in group_results)
        base_record["time_taken"] = total_time
        
        if "chars_per_second" in group_results[0]:
            # Calculate average processing speed
            speeds = [r["chars_per_second"] for r in group_results if "chars_per_second" in r]
            base_record["chars_per_second"] = np.mean(speeds) if speeds else 0
        
        # Task-specific merging
        if task == "dialogue_detection":
            dialogues = []
            total_count = 0
            
            for r in group_results:
                if "count" in r and "sample_dialogues" in r:
                    total_count += r["count"]
                    for dialogue in r["sample_dialogues"]:
                        if dialogue not in dialogues:
                            dialogues.append(dialogue)
            
            # Remove potential duplicates from overlap
            base_record["count"] = total_count
            base_record["sample_dialogues"] = dialogues[:5]  # Take top 5 samples
            
        elif task == "ner":
            characters = []
            unique_count = 0
            
            for r in group_results:
                if "unique_count" in r and "characters" in r:
                    unique_count += r["unique_count"]
                    for char in r["characters"]:
                        if char not in characters:
                            characters.append(char)
            
            # Deduplicate characters across chunks
            base_record["unique_count"] = len(characters)
            base_record["characters"] = characters[:10]  # Take top 10
            
        elif task == "sentiment":
            sentences = []
            total_count = 0
            
            for r in group_results:
                if "emotional_count" in r and "emotional_sentences" in r:
                    total_count += r["emotional_count"]
                    for sent in r["emotional_sentences"]:
                        if sent not in sentences:
                            sentences.append(sent)
            
            base_record["emotional_count"] = total_count
            base_record["emotional_sentences"] = sentences[:5]  # Top 5 samples
        
        merged_results.append(base_record)
    
    return merged_results


In [ ]:
def process_multiple_samples(samples):
    """
    Process multiple text samples in parallel using ProcessPoolExecutor
    
    Args:
        samples (dict): Dictionary mapping sample_id to sample_text
        
    Returns:
        list: Aggregated results from all samples
    """
    # Start with memory cleanup
    print("Starting fresh run - cleaning up memory...")
    clear_memory()
    
    print(f"Using {MAX_WORKERS} parallel processes with chunk size of {CHUNK_SIZE} chars")
    sample_items = list(samples.items())
    all_results = []
    
    # Process samples in parallel using ProcessPoolExecutor
    with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=init_worker) as executor:
        futures = {executor.submit(process_sample_in_chunks, sample_id, sample_text): sample_id 
                  for sample_id, sample_text in sample_items}
        
        # Create a progress bar with tqdm
        for future in tqdm(as_completed(futures), total=len(futures), 
                          desc="Processing samples", unit="sample"):
            sample_id = futures[future]
            try:
                sample_result = future.result()
                if sample_result:  # Check if result is not empty
                    all_results.extend(sample_result)
                    print(f"✓ Completed: {sample_id} ({len(sample_result)} results)")
                else:
                    print(f"✗ Failed: {sample_id} (no results)")
            except Exception as e:
                print(f"✗ Error: {sample_id} - {e}")
                
    # Final memory cleanup
    print("Processing complete - cleaning up memory...")
    clear_memory()
    
    return all_results


In [ ]:
def load_samples(directory="../data/corpus"):
    """
    Load sample texts from corpus directory
    
    Args:
        directory (str): Path to the corpus directory
        
    Returns:
        dict: Dictionary of sample_id to sample_text
    """
    samples = {}
    print("Loading text samples from directory:", directory)
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".txt"):
                file_path = os.path.join(root, filename)
                category = os.path.basename(os.path.dirname(file_path))
                sample_id = f"{category}/{filename}"
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        samples[sample_id] = f.read()
                        print(f"Loaded: {sample_id}")
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    return samples

def create_ground_truth(sample_id, text, dialogues=None, characters=None, emotions=None):
    """
    Create or update ground truth annotations for a text sample
    
    Args:
        sample_id (str): Identifier for the sample
        text (str): The text content
        dialogues (list): List of dialogue strings
        characters (list): List of character names
        emotions (list): List of emotional sentences
        
    Returns:
        dict: Ground truth data
    """
    ground_truth = {
        "sample_id": sample_id,
        "text": text,
        "dialogues": dialogues or [],
        "characters": characters or [],
        "emotions": emotions or []
    }
    
    # Save to JSON
    os.makedirs("../data/ground_truth", exist_ok=True)
    with open(f"../data/ground_truth/{sample_id.replace('/', '_')}.json", "w") as f:
        json.dump(ground_truth, f, indent=2)
    
    return ground_truth

def load_ground_truth(sample_id):
    """
    Load ground truth annotations for a text sample
    
    Args:
        sample_id (str): Identifier for the sample
        
    Returns:
        dict: Ground truth data or None if not found
    """
    path = f"../data/ground_truth/{sample_id.replace('/', '_')}.json"
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None


In [ ]:
def analyze_and_save_results(all_results, samples):
    """
    Analyze benchmark results and save them to files
    
    Args:
        all_results: List of benchmark result dictionaries
        samples: Dictionary of samples that were processed
        
    Returns:
        DataFrame: Results as a pandas DataFrame
    """
    # Convert results to DataFrame for analysis
    results_df = pd.DataFrame(all_results)
    print("\n===== BENCHMARK RESULTS SUMMARY =====\n")

    # Check if we have results
    if results_df.empty:
        print("No benchmark results available. Please check for errors in the processing.")
        return results_df
    
    # Create summary statistics
    summary = results_df.groupby(['task', 'library']).agg({
        'time_taken': ['mean', 'min', 'max', 'std'],
        'sample_length': ['mean', 'count']
    })
    print(summary)

    # Calculate average performance by library and task
    print("\n===== LIBRARY PERFORMANCE COMPARISON =====\n")
    library_task_performance = results_df.pivot_table(
        index='library', 
        columns='task', 
        values='time_taken', 
        aggfunc='mean'
    )
    print("Average processing time (seconds) by library and task:")
    print(library_task_performance)

    library_performance = results_df.groupby('library')['time_taken'].mean()
    print("\nOverall average processing time (seconds):")
    print(library_performance)

    # Save detailed results
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path("../results")
    results_dir.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

    # Save results as JSON
    result_file = results_dir / f"benchmark_results_{timestamp}.json"
    with open(result_file, "w") as f:
        json.dump({
            "sample_count": len(samples),
            "timestamp": str(datetime.datetime.now()),
            "summary": {
                "library_performance": {lib: time for lib, time in library_performance.items()},
                "task_performance": library_task_performance.to_dict()
            },
            "results": all_results
        }, f, indent=2, default=str)

    print(f"\nDetailed results saved to {result_file}")

    # Create CSV for easier analysis
    csv_file = results_dir / f"benchmark_results_{timestamp}.csv"
    results_df.to_csv(csv_file, index=False)
    print(f"CSV results saved to {csv_file}")
    
    return results_df


In [ ]:
def create_visualizations(results_df, results_dir, timestamp):
    """
    Create and save visualizations of benchmark results
    
    Args:
        results_df: DataFrame with benchmark results
        results_dir: Path object for the results directory
        timestamp: Timestamp string for file naming
    """
    try:
        if not results_df.empty:
            # Set style
            sns.set(style="whitegrid")
            
            # Create a figure with multiple subplots
            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            
            # 1. Processing time by task and library (bar chart)
            time_by_task = results_df.pivot_table(
                index='task', 
                columns='library', 
                values='time_taken', 
                aggfunc='mean'
            )
            time_by_task.plot(kind='bar', ax=axes[0, 0], title='Average Processing Time by Task')
            axes[0, 0].set_ylabel('Time (seconds)')
            axes[0, 0].set_xlabel('Task')
            
            # 2. Processing speed comparison (bar chart)
            speed_data = results_df[results_df['task'] == 'processing_speed']
            if not speed_data.empty:
                sns.barplot(x='library', y='chars_per_second', data=speed_data, ax=axes[0, 1])
                axes[0, 1].set_title('Characters Processed Per Second')
                axes[0, 1].set_ylabel('Chars/second')
                axes[0, 1].set_xlabel('Library')
            
            # 3. NER performance (scatter plot)
            ner_data = results_df[results_df['task'] == 'ner']
            if not ner_data.empty:
                scatter = sns.scatterplot(
                    x='sample_length', 
                    y='time_taken', 
                    hue='library', 
                    size='unique_count', 
                    sizes=(20, 200),
                    data=ner_data, 
                    ax=axes[1, 0]
                )
                axes[1, 0].set_title('NER Performance by Sample Size')
                axes[1, 0].set_xlabel('Sample Length (chars)')
                axes[1, 0].set_ylabel('Processing Time (seconds)')
                # Add legend title if needed
                if scatter.get_legend() is not None:
                    handles, labels = scatter.get_legend_handles_labels()
                    if len(handles) > 2:  # Make sure we have size-related handles
                        axes[1, 0].legend(handles=handles[2:], labels=labels[2:], title="Character Count")
            
            # 4. Time vs. text length for all tasks (line plot)
            sns.lineplot(
                x='sample_length', 
                y='time_taken', 
                hue='library',
                style='task',
                markers=True,
                data=results_df, 
                ax=axes[1, 1]
            )
            axes[1, 1].set_title('Processing Time vs. Text Length')
            axes[1, 1].set_xlabel('Sample Length (chars)')
            axes[1, 1].set_ylabel('Time (seconds)')
            
            plt.tight_layout()
            
            # Save the figure
            plot_file = results_dir / f"benchmark_plots_{timestamp}.png"
            plt.savefig(plot_file, dpi=300)
            print(f"Visualization saved to {plot_file}")
            
            # Additional visualizations
            # 1. Box plot of processing times
            plt.figure(figsize=(12, 6))
            sns.boxplot(x='task', y='time_taken', hue='library', data=results_df)
            plt.title('Distribution of Processing Times by Task and Library')
            plt.ylabel('Time (seconds)')
            plt.tight_layout()
            box_plot_file = results_dir / f"benchmark_boxplot_{timestamp}.png"
            plt.savefig(box_plot_file, dpi=300)
            print(f"Box plot saved to {box_plot_file}")
            
            # 2. Heatmap of time correlation between tasks
            plt.figure(figsize=(10, 8))
            # Pivot and calculate correlation
            task_pivot = results_df.pivot_table(
                index=['sample_id', 'library'], 
                columns='task', 
                values='time_taken'
            )
            corr = task_pivot.corr()
            sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
            plt.title('Correlation of Processing Times Between Tasks')
            plt.tight_layout()
            corr_plot_file = results_dir / f"benchmark_correlation_{timestamp}.png"
            plt.savefig(corr_plot_file, dpi=300)
            print(f"Correlation plot saved to {corr_plot_file}")
            
            # 3. Direct comparison of libraries
            plt.figure(figsize=(10, 6))
            comp_data = results_df.pivot_table(
                index='sample_id',
                columns='library',
                values='time_taken',
                aggfunc='sum'
            )
            # Add a 45-degree line to show equal performance
            max_val = max(comp_data.max().max(), 1)
            plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='Equal performance')
            
            # Plot scatter plot of spaCy vs NLTK performance
            if 'spacy' in comp_data.columns and 'nltk' in comp_data.columns:
                plt.scatter(comp_data['spacy'], comp_data['nltk'], alpha=0.7)
                plt.xlabel('spaCy Processing Time (seconds)')
                plt.ylabel('NLTK Processing Time (seconds)')
                plt.title('Direct Comparison: spaCy vs NLTK Performance')
                # Add text labels for points that are far from the equal performance line
                for idx, row in comp_data.iterrows():
                    if abs(row['spacy'] - row['nltk']) > max(row['spacy'], row['nltk']) * 0.5:
                        plt.text(row['spacy'], row['nltk'], idx.split('/')[-1], fontsize=8)
                plt.tight_layout()
                comp_plot_file = results_dir / f"library_comparison_{timestamp}.png"
                plt.savefig(comp_plot_file, dpi=300)
                print(f"Library comparison plot saved to {comp_plot_file}")
            
            # Display all plots
            plt.show()
            
        else:
            print("No data available for visualization. Check for errors in the processing.")
    except Exception as e:
        print(f"Could not create visualizations: {e}")
        import traceback
        traceback.print_exc()

    print("\nBenchmark visualization completed!")


In [ ]:
def validate_against_ground_truth(results_df, samples):
    """
    Compare benchmark results against ground truth data
    
    Args:
        results_df: DataFrame with benchmark results
        samples: Dictionary of samples that were processed
        
    Returns:
        DataFrame with validation metrics
    """
    validation_results = []
    
    for sample_id in samples.keys():
        gt = load_ground_truth(sample_id)
        if not gt:
            continue
            
        # Get results for this sample
        sample_results = results_df[results_df['sample_id'] == sample_id]
        
        for _, result in sample_results.iterrows():
            if result['task'] == 'ner' and 'characters' in result:
                # Compare detected characters with ground truth
                gt_chars = set(gt.get('characters', []))
                detected_chars = set(result['characters'])
                
                # Calculate metrics
                true_positives = len(gt_chars.intersection(detected_chars))
                false_positives = len(detected_chars - gt_chars)
                false_negatives = len(gt_chars - detected_chars)
                
                precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
                recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                
                validation_results.append({
                    'sample_id': sample_id,
                    'task': 'ner',
                    'library': result['library'],
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'true_positives': true_positives,
                    'false_positives': false_positives,
                    'false_negatives': false_negatives
                })
                
            # Similar logic for other tasks (dialogue_detection, sentiment)
            # ...
    
    # Convert to DataFrame
    validation_df = pd.DataFrame(validation_results)
    
    # Save validation results
    if not validation_df.empty:
        validation_df.to_csv(results_dir / f"validation_results_{timestamp}.csv", index=False)
        print(f"Validation results saved to {results_dir / f'validation_results_{timestamp}.csv'}")
        
        # Create validation visualization
        plt.figure(figsize=(12, 8))
        sns.barplot(x='library', y='f1', hue='task', data=validation_df)
        plt.title('F1 Score by Library and Task')
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.savefig(results_dir / f"validation_f1_{timestamp}.png", dpi=300)
        print(f"Validation visualization saved to {results_dir / f'validation_f1_{timestamp}.png'}")
        
    return validation_df


In [ ]:
# Load samples from disk
samples = load_samples(directory="../data/corpus")

# Display sample information
print(f"Loaded {len(samples)} text samples for processing")
for sample_id, text in list(samples.items())[:3]:
    print(f"- {sample_id}: {len(text)} chars")
    gt = load_ground_truth(sample_id)
    if gt:
        print(f"  * Has ground truth data")

# Process the samples
print(f"Starting benchmark with {len(samples)} samples...")
all_results = process_multiple_samples(samples)

# Analyze and save results
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

results_df = analyze_and_save_results(all_results, samples)


In [ ]:
# Generate visualizations if results are available
if 'results_df' in locals() and not results_df.empty:
    print("\nGenerating visualizations...")
    create_visualizations(results_df, results_dir, timestamp)
else:
    print("\nNo results available for visualization")


In [ ]:
# Run validation if ground truth is available
if 'results_df' in locals() and not results_df.empty:
    print("\nValidating results against ground truth...")
    validation_df = validate_against_ground_truth(results_df, samples)
    
    if 'validation_df' in locals() and not validation_df.empty:
        print("\nValidation summary:")
        print(validation_df.groupby(['task', 'library']).mean()[['precision', 'recall', 'f1']])
    else:
        print("No ground truth data available for validation")
        
print("\nBenchmark process complete!")
